![NVIDIA Logo](images/nvidia.png)

# Exercise: Broadcast Three Ways

For this exercise, create a custom passthrough stage which broadcasts to three output ports, then, create a pipeline that hooks the output of your broadcast stage to 2 different in-memory sinks and a single output file.

In [1]:
input_file = 'data/simple_user_log.jsonlines'
output_file = 'data/output/simple_user_log.jsonlines'

---

## Imports

You will likely need to use the following imports in your work.

In [ ]:
import logging
import typing

from IPython.display import Image
import cudf

from morpheus.config import Config

from morpheus.pipeline import Pipeline
from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.single_port_stage import SinglePortStage

from morpheus.pipeline.stage import Stage
from morpheus.pipeline.stage_schema import StageSchema

from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.in_memory_sink_stage import InMemorySinkStage
from morpheus.stages.output.write_to_file_stage import WriteToFileStage

from morpheus.messages import MessageMeta

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
import mrc.core.operators as ops
from mrc.core.node import Broadcast

---

## Your Work Here

Build and run your pipeline in the space provided below. By all means feel free to create additional code cells for your work, which you can do by clicking the `+` button in the Jupyter menu bar at the top of this notebook.

If you get stuck, a solution is provided below, which you view by expanding the *Solution* section below.

---

## Solution

In [ ]:
class MpBroadcastStage(GpuAndCpuMixin, Stage):

    def __init__(self, c: Config):
        super().__init__(c)

        self._create_ports(1, 3) # One input port, two output ports.

    @property
    def name(self) -> str:
        return "mp-bcast"

    def supports_cpp_node(self):
        return False
    
    def accepted_types(self) -> tuple:
        return (MessageMeta, )
    
    def compute_schema(self, schema: StageSchema):
        for port_schema in schema.output_schemas:
            port_schema.set_type(MessageMeta)
    
    def on_data(self, message: MessageMeta) -> MessageMeta:
        return message

    def _build(self, builder: mrc.Builder, input_nodes: list[mrc.SegmentObject]) -> list[mrc.SegmentObject]:

        # Create a broadcast node
        broadcast = Broadcast(builder, "broadcast")
        builder.make_edge(input_nodes[0], broadcast)

        # Here we create 3 outgoing nodes.
        outgoing_1 = builder.make_node("outgoing_1", ops.map(self.on_data))
        outgoing_2 = builder.make_node("outgoing_2", ops.map(self.on_data))
        outgoing_3 = builder.make_node("outgoing_3", ops.map(self.on_data))
        
        # Here we create edges between the broadcast node and each of the outgoing nodes.
        builder.make_edge(broadcast, outgoing_1)
        builder.make_edge(broadcast, outgoing_2)
        builder.make_edge(broadcast, outgoing_3)

        # Here we return all three outgoing nodes.
        return [outgoing_1, outgoing_2, outgoing_3]

In [ ]:
config = Config()
pipeline = Pipeline(config)

source = pipeline.add_stage(FileSourceStage(config, filename=input_file, iterative=False))
broadcast = pipeline.add_stage(MpBroadcastStage(config))
pipeline.add_edge(source, broadcast)

in_mem_sink_1 = pipeline.add_stage(InMemorySinkStage(config))
in_mem_sink_2 = pipeline.add_stage(InMemorySinkStage(config))
write_to_file = pipeline.add_stage(WriteToFileStage(config, filename=output_file, overwrite=True))


pipeline.add_edge(broadcast.output_ports[0], in_mem_sink_1)
pipeline.add_edge(broadcast.output_ports[1], in_mem_sink_2)
pipeline.add_edge(broadcast.output_ports[2], write_to_file)

In [ ]:
pipeline.build()

In [ ]:
viz_file = './pipeline_visualizations/broadcast_three_way_passthrough.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
reset_logging()
configure_logging(log_level=logging.DEBUG)

In [ ]:
await pipeline.run_async()

Since we created two output memory sink stages, let's observe them both.

In [ ]:
messages = in_mem_sink_1.get_messages()
messages[0].df

In [ ]:
messages = in_mem_sink_2.get_messages()
messages[0].df

In [ ]:
cudf.read_json(output_file, lines=True)